In [0]:
%sql

-- 1. Number of records in layers
SELECT 'bronze' AS layer, COUNT(*) AS row_count
FROM dbr_dev_ua5816bd.team_crypto_bronze_live_test.ticks_bronze
UNION ALL
SELECT 'silver', COUNT(*)
FROM dbr_dev_ua5816bd.team_crypto_silver_live_test.ticks_silver
UNION ALL
SELECT 'quarantine', COUNT(*)
FROM dbr_dev_ua5816bd.team_crypto_silver_live_test.ticks_quarantine;

In [0]:
%sql
-- 2. Invalid records in Silver
-- Expected: 0 rows
SELECT *
FROM dbr_dev_ua5816bd.team_crypto_silver_live_test.ticks_silver
WHERE symbol IS NULL
   OR symbol NOT IN ('BTC', 'ETH', 'SOL')
   OR event_time IS NULL
   OR price_usd IS NULL
   OR isnan(price_usd)
   OR price_usd <= 0
   OR price_usd >= CAST('Infinity' AS DOUBLE)
   OR ingestion_timestamp IS NULL
   OR event_time > ingestion_timestamp + INTERVAL 5 MINUTES;

In [0]:
%sql

-- 3. Duplicates 
-- Expected: 0 rows
SELECT symbol, event_time, source, COUNT(*) AS copies
FROM dbr_dev_ua5816bd.team_crypto_silver_live_test.ticks_silver
GROUP BY symbol, event_time, source
HAVING COUNT(*) > 1;


In [0]:
%sql
-- 4. Viewing rejected messages and reasons
SELECT value, failed_rules, quarantine_reason
FROM dbr_dev_ua5816bd.team_crypto_silver_live_test.ticks_quarantine;

In [0]:
%sql
-- 5. Rejected records without explanation.
-- Expected: 0 rows.

SELECT *
FROM dbr_dev_ua5816bd.team_crypto_silver_live_test.ticks_quarantine
WHERE failed_rules IS NULL
   OR size(failed_rules) = 0
   OR quarantine_reason IS NULL
   OR trim(quarantine_reason) = '';